# 실습

상환 여부를 예측하기 위한 의사결정나무 분석을 실시하고 결과를 해석하라.

분석 절차: 
* [데이터 구성하기](#데이터-구성하기)
* [의사결정나무 모델 생성](#의사결정나무-모델-생성)
* [최종 모델 선정, 시각화](#최종-모델-선정,-시각화)
* [결론 도출](#결론-도출)


[Help](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier)


#### 패키지 불러오기

In [ ]:
# 데이터 구성:Series, DataFrame
import pandas as pd
import numpy as np
# 데이터 시각화
import matplotlib.pyplot as plt
import matplotlib
# export_graphviz: 나무 구조 생성 및 저장 
from sklearn.tree import export_graphviz
# graphviz : 나무 구조 시각화  (.dot 확장자 파일 불러오기 등)
import graphviz

# 다른 방식(.dot -> .png 형식, 출력화면에 맞는)으로 Tree 출력
from subprocess import call
from IPython.display import Image

# 데이터 분할:train, test
from sklearn.model_selection import train_test_split
# 분류 Decision Tree
from sklearn.tree import DecisionTreeClassifier
# 최적 모델, 파라미터 탐색
from sklearn.model_selection import GridSearchCV

# 분류모델 평가 함수
from sklearn.metrics import accuracy_score, f1_score 
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Tree 생성경로 지정
import os
# PATH 설정: graphviz를 설치 했다면, 설치 된 경로를 설정. 기본 경로는 아래 예제 참고(linux에서 설치된 경로 확인 및 변경 필요)
# os.environ["PATH"] += os.pathsep + "C:/Program Files (x86)/Graphviz2.38/bin/"
os.environ["PATH"] += os.pathsep + "C:/Program Files/Graphviz/bin/"

#### 그래프 옵션 지정

In [ ]:
# 그래프 한글폰트 적용:맑은 고딕
matplotlib.rc("font", family = "Malgun Gothic")
# 그래프 (-) 기호 표시
matplotlib.rc("axes", unicode_minus = False)

## 데이터 구성하기

#### 데이터 불러오기

In [ ]:
df_raw = pd.read_csv("D:/WORK/DATA/통신고객이탈.CSV" , engine = "python")
df_raw.head()

In [ ]:
# Data 구조 확인
print("Data 구조:", df_raw.shape)
print()
print("변수 : ", df_raw.columns)

#### 결측치 확인

In [ ]:
df_raw.isnull().sum(axis = 0)

#  결측처리 불필요

#### 범주형 설명변수 더미 변환

In [ ]:
# drop: X변수외 변수 삭제
df_raw_x = df_raw.drop(["CHURN","CUSTOMER_ID"], axis =1, inplace = False)

# get_dummies: 데이터의 문자형 변수에 대한 더미변수 생성 
df_raw_dummy = pd.get_dummies(df_raw_x)
df_raw_dummy.head()

#### 데이터 분리

In [ ]:
# 데이터 분리:설명변수, 목표변수 구분
df_raw_y = df_raw["CHURN"] 
df_x_dummy = pd.get_dummies(df_raw_x)
df_raw_y = np.where(df_raw_y == "Active", 0, 1)

#### train, test 데이터 분할

In [ ]:
# train_test_split(X: 설명변수 데이터, Y: 목표변수 데이터, test_size = test 데이터 비율, random_state: 랜덤)
df_train_x, df_test_x, df_train_y, df_test_y = train_test_split(df_x_dummy, # 설명변수 데이터
                                                                df_raw_y, # 목표변수 데이터
                                                                test_size = 0.3, # test 데이터의 비율
                                                                random_state = 1234)  # random state

print("train data X size : {}".format(df_train_x.shape))
print("train data Y size : {}".format(df_train_y.shape))
print("test data X size : {}".format(df_test_x.shape))
print("test data Y size : {}".format(df_test_y.shape))

### @불균형 자료 사전 처리:over-, under-sampling-SMOTE
class imblearn.over_sampling.SMOTE(*, sampling_strategy='auto', random_state=None, k_neighbors=5, n_jobs=None)

In [1]:
# 샘플링 : Over-sampling 등
from imblearn.over_sampling import SMOTE

In [2]:
# 목표변수 빈도 확인
print(df_raw.value_counts(["CHURN"]),"\n")
# print(df_raw["BAD"].value_counts(), "\n")
print("CHURN=Churned 비율  ", df_raw.value_counts(df_raw["CHURN"]=="Churned")/len(df_raw))

# 목표변수 산점도 확인
plt.figure(figsize=(10,8))

df_raw['color'] = np.where(df_raw["CHURN"]=="Churned", "red", "blue")
plt.scatter(df_raw['SERVICE_DURATION'],df_raw['DROPPED_CALLS'], c=df_raw['color'], alpha=0.5)
plt.show()

NameError: name 'df_raw' is not defined

In [ ]:
# Over-sampling 설정
sm = SMOTE(sampling_strategy='auto')

# train데이터를 이용한 Over-sampling : x_resampled-df, y_resampled-array
x_resampled, y_resampled = sm.fit_resample(df_train_x,df_train_y)

# 결과 확인
# print('Over-Sampling 전:\n',df_train_y.value_counts(),"\n")
print('Over-Sampling 후 Train X: {}'.format(x_resampled.shape))
print('Over-Sampling 후 Train Y: {} \n'.format(y_resampled.shape))

print("Over-Sampling 후 '1, Churned':{}".format(sum(y_resampled==1)))
print("Over-Sampling 후 '0, Active':{}".format(sum(y_resampled==0)))

In [ ]:
# 데이터 결합 및 산점도 확인
y_resampled_series = pd.Series(y_resampled)

In [ ]:
df_resampled = pd.concat([x_resampled,y_resampled_series], axis=1)
# df_resampled = pd.concat([x_resampled,y_resampled], axis=1)
print(df_resampled.head())

In [ ]:
df_resampled.dtypes

In [ ]:
# 목표변수 산점도 확인
plt.figure(figsize=(10,8))
plt.scatter(df_resampled['SERVICE_DURATION'],df_resampled['DROPPED_CALLS']
            ,c=df_resampled[0],alpha=0.5)  # CHURN -> df_resampled[0]
plt.show()

## 의사결정나무 모델 생성

#### default parameter로 모델 생성

In [ ]:
tree_uncust = DecisionTreeClassifier(random_state=1234)
tree_uncust.fit(df_train_x, df_train_y)
# 훈련 데이터 정확도
print("Accucary on training set: {:.3f}".format(tree_uncust.score(df_train_x, df_train_y)))
# test 데이터 정확도
print("Accucary on test set: {:.3f}".format(tree_uncust.score(df_test_x, df_test_y)))

In [ ]:
tree_uncust

#### max_depth: 최대 깊이 변경에 따른 정확도 변화

In [ ]:
# train 및 test 정확도 결과 저장용
train_accuracy = []; test_accuracy = []
# max_depth: 최대 깊이 변경
para_depth = [depth for depth in range(3, 11)]

for max_depth in para_depth:
    tree = DecisionTreeClassifier(max_depth = max_depth, random_state=1234)
    tree.fit(df_train_x, df_train_y)
    train_accuracy.append(tree.score(df_train_x, df_train_y))
    test_accuracy.append(tree.score(df_test_x, df_test_y))

# 데이터 테이블로 저장
df_accuracy_depth = pd.DataFrame()
df_accuracy_depth["Depth"] = para_depth
df_accuracy_depth["TrainAccuracy"] = train_accuracy
df_accuracy_depth["TestAccuracy"] = test_accuracy
df_accuracy_depth.round(3)

In [ ]:
# 정확도를 그래프로 표현
plt.plot(para_depth, train_accuracy, linestyle = "-", label = "Train Accuracy")
plt.plot(para_depth, test_accuracy, linestyle = "--", label = "Test Accuracy")
plt.legend()

#### 깊이(max_depth)에 따른 차이 변화(깊이 4 vs 5)

In [ ]:
# 변수명
feature_names = df_train_x.columns
# 깊이:얕은 모델
tree_depth4 = DecisionTreeClassifier(max_depth = 4, random_state=1234)
tree_depth4.fit(df_train_x, df_train_y)
# 트리 모델을 tree_depth4.dot 파일로 저장. (목표변수, 0: Good, 1: Bad)
export_graphviz(tree_depth4, out_file="tree_low.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)
# graphviz를 이용해 트리 모델 시각화
with open("tree_low.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

In [ ]:
# 생성된 .dot 파일을 .png로 변환
call(['dot', '-Tpng', 'tree_low.dot', '-o', 'tree_low.png', '-Gdpi=600'])
# jupyter notebook에서 작업디렉토리에 있는 .png 직접 출력
Image(filename = 'tree_low.png')

In [ ]:
# 깊이:깊은 모델
tree_depth6 = DecisionTreeClassifier(max_depth =5, random_state=1234)
tree_depth6.fit(df_train_x, df_train_y)
export_graphviz(tree_depth6, out_file="tree_high.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)
with open("tree_high.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

#### min_samples_leaf: 잎사귀 노드의 샘플 수 제한

In [ ]:
# train 및 test 정확도 결과 저장용
train_accuracy = []; test_accuracy = []
# min_samples_leaf: 잎사귀 수 제한
para_leaf = [n_leaf * 5 for n_leaf in range(3, 11)]

for min_samples_leaf in para_leaf:
    tree = DecisionTreeClassifier(min_samples_leaf=min_samples_leaf, max_depth=5, random_state=1234)
    tree.fit(df_train_x, df_train_y)
    train_accuracy.append(tree.score(df_train_x, df_train_y))
    test_accuracy.append(tree.score(df_test_x, df_test_y))

# 데이터 테이블로 저장
df_accuracy_leaf = pd.DataFrame()
df_accuracy_leaf["MinSamplesLeaf"] = para_leaf
df_accuracy_leaf["TrainAccuracy"] = train_accuracy
df_accuracy_leaf["TestAccuracy"] = test_accuracy
df_accuracy_leaf.round(3)

In [ ]:
# 정확도를 그래프로 표현
plt.plot(para_leaf, train_accuracy, linestyle = "-", label = "Train Accuracy")
plt.plot(para_leaf, test_accuracy, linestyle = "--", label = "Test Accuracy")
plt.legend()

#### 잎사귀 노드의 샘플 수(min_samples_leaf)를 변경하면서 모델의 시각화 결과를 확인

In [ ]:
# 잎사귀 노드의 샘플 수가 20인 모델
tree_leaf20 = DecisionTreeClassifier(max_depth =5, min_samples_leaf = 20, random_state=1234)
tree_leaf20.fit(df_train_x, df_train_y)

export_graphviz(tree_leaf20, out_file="tree.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)

with open("tree.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

min_samples_leaf = 60

In [ ]:
# 잎사귀 노드의 샘플 수가 60인 모델
tree_leaf60 = DecisionTreeClassifier(max_depth = 4, min_samples_leaf = 60, random_state=1234)
tree_leaf60.fit(df_train_x, df_train_y)

export_graphviz(tree_leaf60, out_file="tree.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)

with open("tree.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

#### min_samples_split: 분할하기 위한 노드의 최소 샘플 수

In [ ]:
# train 및 test 정확도 결과 저장용
train_accuracy = []; test_accuracy = []
# min_samples_split: 분할하기 위한 노드의 최소 샘플 수 
para_split = [n_split * 20 for n_split in range(1, 11)]

for min_samples_split in para_split:
    tree = DecisionTreeClassifier(min_samples_split=min_samples_split, min_samples_leaf = 20, max_depth =5, random_state=1234)
    tree.fit(df_train_x, df_train_y)
    train_accuracy.append(tree.score(df_train_x, df_train_y))
    test_accuracy.append(tree.score(df_test_x, df_test_y))

# 데이터 테이블로 저장
df_accuracy_split = pd.DataFrame()
df_accuracy_split["MinSamplesSplit"] = para_split
df_accuracy_split["TrainAccuracy"] = train_accuracy
df_accuracy_split["TestAccuracy"] = test_accuracy
df_accuracy_split.round(3)

In [ ]:
# 정확도를 그래프로 표현
plt.plot(para_split, train_accuracy, linestyle = "-", label = "Train Accuracy")
plt.plot(para_split, test_accuracy, linestyle = "--", label = "Test Accuracy")
plt.legend()

## 최종 모델 선정, 시각화

#### 최종 모델

In [ ]:
tree_final = DecisionTreeClassifier(max_depth = 5, min_samples_leaf = 20)
tree_final.fit(df_train_x, df_train_y)

In [ ]:
# 평가
y_pred = tree_final.predict(df_test_x)
print("Accuracy: {0:.3f}\n".format(tree_final.score(df_test_x, df_test_y)))
print("Confusion matrix: \n{}".format(confusion_matrix(df_test_y, y_pred)))

# 목표변수의 빈도 불균형 : f1 score로 모델 평가 
print(classification_report(df_test_y, y_pred, digits=3))

In [ ]:
# tree_final.dot으로 결과 저장
export_graphviz(tree_final, out_file="tree.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)
# tree_final.dot 그리기
with open("tree.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

## 결론 도출

In [ ]:
# tree.feature_importances_로 설명변수 중요도 확인 및 테이블로 저장
df_importance = pd.DataFrame()
df_importance["Feature"] = feature_names
df_importance["Importance"] = tree_final.feature_importances_

# ds_feature_importance의 테이블을 중요도별로 정렬
df_importance.sort_values("Importance", ascending=False, inplace = True)
df_importance.round(3)

In [ ]:
# 설명변수 중요도 그래프
# 중요도가 높은 변수를 상위에 그림. 
df_importance.sort_values("Importance", ascending=True, inplace = True)
coordinates = range(len(df_importance))
plt.barh(y = coordinates, width = df_importance["Importance"])
plt.yticks(coordinates, df_importance["Feature"])
plt.xlabel("설명변수 중요도")
plt.ylabel("설명변수")


### @추가연습:Over-sampling Data이용한 모델링 및 평가

In [ ]:
# 최종 모델의 hyper-parameter 이용
tree_final = DecisionTreeClassifier(max_depth = 5, min_samples_leaf = 20)

# Over-sampling Data 지정
tree_final.fit(x_resampled, y_resampled)

In [ ]:
# 평가
y_pred = tree_final.predict(df_test_x)
print("Accuracy: {0:.3f}\n".format(tree_final.score(df_test_x, df_test_y)))
print("Confusion matrix: \n{}".format(confusion_matrix(df_test_y, y_pred)))

# 목표변수의 빈도 불균형 : f1 score로 모델 평가 
print(classification_report(df_test_y, y_pred, digits=3))

In [ ]:
# tree_final.dot으로 결과 저장
export_graphviz(tree_final, out_file="tree.dot", class_names = ["Active", "Churned"],
                feature_names = feature_names, impurity = True, filled = True)
# tree_final.dot 그리기
with open("tree.dot") as f:
    dot_graph = f.read()
display(graphviz.Source(dot_graph))

## 결론 도출

In [ ]:
# tree.feature_importances_로 설명변수 중요도 확인 및 테이블로 저장
df_importance = pd.DataFrame()
df_importance["Feature"] = feature_names
df_importance["Importance"] = tree_final.feature_importances_

# ds_feature_importance의 테이블을 중요도별로 정렬
df_importance.sort_values("Importance", ascending=False, inplace = True)
df_importance.round(3)

In [ ]:
# 설명변수 중요도 그래프
# 중요도가 높은 변수를 상위에 그림. 
df_importance.sort_values("Importance", ascending=True, inplace = True)
coordinates = range(len(df_importance))
plt.barh(y = coordinates, width = df_importance["Importance"])
plt.yticks(coordinates, df_importance["Feature"])
plt.xlabel("설명변수 중요도")
plt.ylabel("설명변수")


# End of Code